# oil-and-gas-semantic — Primer · What's built, and what it shows

The executable twin of [`docs/primer.html`](../docs/primer.html). It walks every layer that was built and
deployed, and runs each one so you can see it work, not just read about it:

It is in three parts:

**Part I · What the system contains**: the files in the repo and what the generator writes. No platform
is involved, and it all runs locally.

1. **The data model**: nine OSDU Production Data Model tables, and how they relate.
2. **The records**: generated deterministically, with the answers to every catalog question planted in
   them and written out as gold.
3. **Governed metrics**: one definition of each KPI, compiled to SQL, which reproduces gold.
4. **The ontology**: the OWL classes and keys, the R2RML mapping, the SKOS vocabulary, and the
   **reasoning in action** (inheritance, roll-up, inverses, identity).
5. **Rules**: the business term *below expected*, as data and as a SHACL rule, the two agreeing.
6. **Constraints**: SHACL shapes that find planted data-quality faults, matching gold.

**Part II · One question, end to end**

7. **The hero question**, followed from the catalog through the generated records, the metric, the rule
   and gold, to a graded answer.

**Part III · How it's instantiated**: the same contents, deployed and asked live.

8. **A second engine**: GraphDB gives the same answers, and Ontop serves the daily volumes in place.
9. **The knowledge, generated onto the platforms**: Snowflake and Fabric views generated from the
   ontology, checked live against it.
10. **The platforms' semantic models**: a Snowflake Semantic View and the Fabric Direct Lake model
    reproduce gold.
11. **The agents**: the ontology agent (Claude), the Snowflake Cortex Agent and the Fabric custom agent,
    asked live and graded against gold.

> Everything runs live. Local sections need only `uv sync`. GraphDB needs `knowledge/graphdb/graphdb.sh up`;
> Snowflake needs the `snow` connection `proton`; Fabric needs `az login`, the `FABRIC_*` variables and a
> running capacity; the agents need `ANTHROPIC_API_KEY` and `OAG_FOUNDRY_ENDPOINT`. Every variable is listed in
> [`.env.example`](../.env.example), with where its value lives. A section whose runtime isn't there says so and
> is skipped. No credentials live in the repo or in this notebook.

## 0 · Setup

The reference dataset is `data/frozen` (`config_hash 12a110eecfe2`): 3 fields, 18 wells, 6 facilities,
3 operators, and over a year of daily volumes. `show()` renders rows as a table; `runtimes` says what is reachable.

In [1]:
import json, logging, os, sys, tempfile, warnings
warnings.filterwarnings('ignore'); logging.disable(logging.WARNING)  # connector chatter, not results
from pathlib import Path
import duckdb, pandas as pd

REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path[:0] = [str(REPO / 'src'), str(REPO / 'platforms' / 'fabric'), str(REPO / 'platforms' / 'fabric' / 'agent')]
DATA = REPO / 'data' / 'frozen'
GOLD = DATA / 'gold'
gold = lambda name: json.loads((GOLD / f'{name}.json').read_text())
pd.set_option('display.max_colwidth', 90)

def show(rows, n=None):
    df = pd.DataFrame(rows)
    return df if n is None else df.head(n)

from oag_semantic import graphdb
runtimes = {
    'GraphDB': graphdb.available(),
    'Snowflake (snow connection)': Path('~/Library/Application Support/snowflake/config.toml').expanduser().exists(),
    'Fabric (FABRIC_* env)': bool(os.environ.get('FABRIC_WORKSPACE_ID')),
    'Claude (ANTHROPIC_API_KEY)': bool(os.environ.get('ANTHROPIC_API_KEY')),
    'Foundry (OAG_FOUNDRY_ENDPOINT)': bool(os.environ.get('OAG_FOUNDRY_ENDPOINT')),
}
print('dataset:', json.loads((DATA / 'dataset.json').read_text())['config_hash'])
show([{'runtime': k, 'available': v} for k, v in runtimes.items()])

dataset: 12a110eecfe2


,runtime,available
0,GraphDB,True
1,Snowflake (snow connection),True
2,Fabric (FABRIC_* env),False
3,Claude (ANTHROPIC_API_KEY),False
4,Foundry (OAG_FOUNDRY_ENDPOINT),False


# Part I · What the system contains

Everything in this part is files in the repo and what the generator writes from them: the data model,
the records and their gold answers, the metric definitions, the ontology, its rules and its constraints.
No platform is involved; it all runs in this kernel.

## 1 · The data model: nine canonical tables  <code>oag_generator.schema · spec/osdu</code>

The system of record is nine tables from the OSDU Production Data Model, with OSDU's table and column
names. Every platform loads these same tables.

![The canonical data model: nine OSDU PDM tables and their keys](../docs/diagrams/canonical-data-model.svg)

*Source: [`docs/diagrams/canonical-data-model.drawio`](../docs/diagrams/canonical-data-model.drawio).*

They fall into three groups:

- **Master data** (blue): fields, facilities and wells. A well belongs to a field and flows to a
  facility (a battery). A facility is keyed by the pair `(FACILITY_ID, FACILITY_TYPE)`, as in the PDM.
  A well's `UWI` is an alternate key.
- **Reporting entities** (purple): the PDM's indirection for anything volumes are reported against.
  Each row points at a well or a field (`REPORTING_ENTITY_KIND` says which). Allocation factors run from
  a field's reporting entity to its wells'.
- **Facts** (green): daily measured volumes and well tests hang off the well directly. The forecast
  volumes (`PRODUCT_VOLUME_SUMMARY`, `QUANTITY_METHOD = 'Forecast'`) and downtime events hang off the
  well's *reporting entity*, so reaching them from a well means going through `REPORTING_ENTITY`.

DuckDB reads the Parquet in place:

In [2]:
from oag_generator import canonical_table_paths
paths = canonical_table_paths(DATA)
con = duckdb.connect()
for name, path in paths.items():
    con.execute(f"CREATE VIEW {name} AS SELECT * FROM read_parquet('{path}')")
con.sql(' UNION ALL '.join(f"SELECT '{n.upper()}' AS table_name, COUNT(*) AS n_rows FROM {n}" for n in paths)
        + ' ORDER BY n_rows DESC').df()

,table_name,n_rows
0,PRODUCT_VOLUME_SUMMARY,8208
1,WELL_VOL_DAILY,8208
2,DOWN_TIME_EVENT,282
3,WELL_TEST,251
4,REPORTING_ENTITY,21
5,RPEN_ALLOCATION_FACTOR,18
6,WELL,18
7,FACILITY,6
8,FIELD,3


The master data the ontology is built from: wells (with their UWI, field, facility and operator),
fields and facilities. Facilities are keyed by the pair `(FACILITY_ID, FACILITY_TYPE)`, as in the OSDU PDM.

In [3]:
con.sql('''SELECT w.WELL_ID, w.UWI, w.WELL_NAME, f.FIELD_NAME, w.FACILITY_ID, w.OPERATOR
           FROM well w JOIN field f USING (FIELD_ID) ORDER BY w.WELL_ID''').df()

,WELL_ID,UWI,WELL_NAME,FIELD_NAME,FACILITY_ID,OPERATOR
0,1,NO 15/9-F-1,Volve-1,Volve,1,Equinor
1,2,NO 15/9-F-2,Volve-2,Volve,2,Equinor
2,3,NO 15/9-F-3,Volve-3,Volve,1,Equinor
3,4,NO 15/9-F-4,Volve-4,Volve,2,Equinor
4,5,NO 15/9-F-5,Volve-5,Volve,1,Equinor
5,6,NO 15/9-F-6,Volve-6,Volve,2,Equinor
6,7,NO 15/9-F-7,Sleipner-1,Sleipner,3,AkerBP
7,8,NO 15/9-F-8,Sleipner-2,Sleipner,4,AkerBP
8,9,NO 15/9-F-9,Sleipner-3,Sleipner,3,AkerBP
9,10,NO 15/9-F-10,Sleipner-4,Sleipner,4,AkerBP


## 2 · The records: generated, with the answers planted in  <code>oag_generator · data/frozen</code>

The generator writes the records from a config, deterministically: the same config gives the same bytes,
stamped with a `config_hash`. The config is a set of scenario knobs: how wells decline, how many are
impaired, how often they go down and why, which wells break through, which tests go stale.

Because the generator decides what is true, it also writes down the answer to every catalog question
as **gold**, alongside the tables. Gold is what every runtime in this project is graded against.

In [4]:
manifest = json.loads((DATA / 'dataset.json').read_text())
cfg = manifest['config']
print(f"config_hash {manifest['config_hash']} · seed {cfg['seed']} · {cfg['start_date']} .. {cfg['end_date']}")
print(f"{cfg['n_fields']} fields × {cfg['wells_per_field']} wells, operators {cfg['operators']}")
print('performance:', cfg['performance'])
print('breakthrough fraction:', cfg['breakthrough']['fraction'], '· downtime events per well-year:', cfg['downtime']['events_per_well_year'])
print('surveillance: window', cfg['surveillance_window_days'], 'days, flag below',
      cfg['surveillance_flag_threshold'], 'of expected')
from oag_generator.questions import load_catalog
catalog_text = {q.id: q.text for q in load_catalog().questions()}
show([{'gold file': p.name, 'question_id': g['question_id'], 'catalog question': catalog_text.get(g['question_id'], '')[:90]}
      for p in sorted(GOLD.glob('*.json')) for g in [json.loads(p.read_text())]])

config_hash 12a110eecfe2 · seed 42 · 2024-01-01 .. 2025-03-31
3 fields × 6 wells, operators ['Equinor', 'AkerBP', 'Wintershall']
performance: {'bias_mean': 1.0, 'bias_sd': 0.04, 'ceil': 1.3, 'daily_noise_sd': 0.05, 'floor': 0.0, 'impaired_bias_mean': 0.72, 'impaired_bias_sd': 0.08, 'impaired_fraction': 0.2}
breakthrough fraction: 0.35 · downtime events per well-year: 12.0
surveillance: window 7 days, flag below 0.9 of expected


,gold file,question_id,catalog question
0,decline.json,decline-12-month-field-oil,"What is the 12-month oil decline for Field X, and which wells are declining faster tha..."
1,deferment.json,deferment-top-downtime-causes,"What did we defer last month, and what were the top downtime causes?"
2,knowledge_assets.json,knowledge-assets-by-field,"List every asset in each field, wells and facilities alike, with the field it belongs to."
3,knowledge_identity.json,knowledge-osdu-record-identity,Which canonical well or facility does each OSDU well and facility record identify?
4,knowledge_liquids.json,knowledge-gross-liquids-by-field,How much gross liquids did each field produce this month?
5,rollups.json,rollup-oil-gas-water-by-field-operator,Oil/gas/water by field and operator this month vs last -- who are the biggest movers?
6,surveillance.json,surveillance-below-expected-oil,"Which wells produced below expected oil rate this week, and by how much?"
7,watchlist.json,watchlist-down-watering-out-gor-change,"Which wells are down, watering out, or showing a GOR change?"
8,welltest.json,welltest-stale-or-anomalous-allocation,Which wells have stale tests or anomalous allocation?


## 3 · Governed metrics: one definition, compiled  <code>semantic/ · oag_semantic.compile</code>

Each KPI is defined once, in an OSI-style YAML semantic layer: measures over the canonical tables, and
metrics built from them. *Production efficiency* is a ratio of two governed metrics. Platforms translate
these definitions; the reference compile runs them on DuckDB.

In [5]:
import yaml
docs = [d['metric'] for d in yaml.safe_load_all((REPO / 'semantic' / 'metrics.yaml').read_text()) if d and 'metric' in d]
print(yaml.safe_dump([d for d in docs if d['name'] in ('actual_oil', 'expected_oil', 'production_efficiency')],
                     sort_keys=False))

- name: actual_oil
  description: Total measured oil produced over the window (bbl).
  type: simple
  type_params:
    measure: actual_oil_volume
- name: expected_oil
  description: Total forecast (expected) oil over the window (bbl) -- the decline-curve
    baseline.
  type: simple
  type_params:
    measure: expected_oil_volume
- name: production_efficiency
  description: actual oil / expected oil; the surveillance signal ("below expected"
    = below the materiality band).
  type: ratio
  type_params:
    numerator:
      name: actual_oil
    denominator:
      name: expected_oil



**The hero question**: *which wells produced below expected oil this week, and by how much?* The compile
answers it from the governed metrics, and the answer equals gold, the generator's own record of which
wells it made under-perform.

In [6]:
from oag_semantic.compile import compute_surveillance
surv = compute_surveillance(DATA)
flagged = surv.flagged
want = sorted(r['well_id'] for r in gold('surveillance')['flagged'])
print(f'window {surv.window_start} .. {surv.window_end}; flagged {sorted(w.well_id for w in flagged)}; gold {want}')
show([{'well_id': w.well_id, 'expected_oil_bbl': round(w.expected_oil_bbl, 1), 'actual_oil_bbl': round(w.actual_oil_bbl, 1),
       'efficiency': round(w.efficiency, 3)} for w in flagged])

window 2025-03-25 .. 2025-03-31; flagged [1, 2, 5, 6, 7, 8, 9, 10, 11, 12, 13, 17]; gold [1, 2, 5, 6, 7, 8, 9, 10, 11, 12, 13, 17]


,well_id,expected_oil_bbl,actual_oil_bbl,efficiency
0,12,82985.1,50794.1,0.612
1,11,108594.0,76954.9,0.709
2,10,66517.2,43479.0,0.654
3,2,39240.4,17117.5,0.436
4,7,68066.4,46380.2,0.681
5,13,59986.5,38548.8,0.643
6,6,12421.4,3961.5,0.319
7,1,12994.2,4686.9,0.361
8,17,65977.5,58073.4,0.880
9,8,10512.9,4934.2,0.469


## 4 · The ontology: meaning the platforms don't carry  <code>knowledge/ontology · oag_semantic.ontology</code>

Standards-based (W3C RDF, OWL 2 RL, SKOS, R2RML). Four files:

| file | what it holds |
|---|---|
| `oag.ttl` | the **TBox**: classes (Asset ⊃ Well, Facility, Field), properties (transitive `partOf`, inverses), **keys** (`owl:hasKey`) aligned to OSDU kinds |
| `mapping.r2rml.ttl` | **R2RML**: canonical rows → instances, so the graph is built from the tables, never typed in |
| `knowledge/vocabulary.yaml` → SKOS | field synonyms, **business terms** (*below expected* → metric, comparison, threshold), the **product hierarchy** |
| `shapes.ttl`, `rules.ttl` | SHACL constraints (§6) and rules (§5) |

The schema in brief: classes and the keys that identify their instances.

In [7]:
from rdflib import Graph
from oag_semantic.ontology import TBOX_PATH, PREFIXES
tbox = Graph().parse(TBOX_PATH)
from rdflib import URIRef
short = lambda v: tbox.qname(URIRef(v)) if v.startswith('http') else v
show([{k: ', '.join(short(x) for x in str(v).split(', ')) for k, v in r.asdict().items()} for r in tbox.query('''
    SELECT ?class ?parent (GROUP_CONCAT(?k; separator=", ") AS ?key) ?osdu_kind WHERE {
      ?class a owl:Class ; oag:osduKind ?osdu_kind . OPTIONAL { ?class rdfs:subClassOf ?parent }
      OPTIONAL { ?class owl:hasKey/rdf:rest*/rdf:first ?k } } GROUP BY ?class ?parent ?osdu_kind ORDER BY ?class''',
    initNs={**PREFIXES, 'rdfs': __import__('rdflib').RDFS})])

,class,parent,key,osdu_kind
0,oag:Facility,oag:Asset,"oag:facilityId, oag:facilityType",osdu:wks:master-data--GenericFacility:1.0.1
1,oag:Field,oag:Asset,,osdu:wks:master-data--Field:1.1.0
2,oag:Organisation,NaN,,osdu:wks:master-data--Organisation:1.2.0
3,oag:Well,oag:Asset,oag:uwi,osdu:wks:master-data--Well:1.5.0


### R2RML builds the instances from the tables

What one well looks like after mapping: its type, its keys, and links **only to its facility and field**.
Nothing says it is an *Asset*, that it is part of its field through the facility, or which OSDU record
it is. Those are inferred next.

In [8]:
from oag_semantic.ontology import ID, OAG, asserted_graph, load_ontology
asserted = asserted_graph(DATA)
well = ID['well/1']
fmt = lambda o: str(o).replace(str(ID), 'id:').replace(str(OAG), 'oag:')
show([{'predicate': asserted.qname(p), 'object': fmt(o)} for p, o in sorted(asserted.predicate_objects(well))])

,predicate,object
0,rdf:type,oag:Well
1,skos:prefLabel,Volve-1
2,oag:atFacility,id:facility/1/Battery
3,oag:inField,id:field/1
4,oag:operatedBy,id:organisation/Equinor
5,oag:uwi,NO 15/9-F-1
6,oag:wellId,1


### Reasoning in action

`load_ontology` runs the OWL 2 RL closure (owlrl) over the asserted graph, with the key rule applied in
code (owlrl's is wrong, ADR 0042). Then each inference below is a fact the graph **derived**, which no
table or platform states: compare *asserted* with *inferred*. Counts are of canonical entities only:
owl:sameAs also copies each fact onto the asset's OSDU-record alias.

In [9]:
onto = load_ontology(DATA)
g = onto.graph
def count(graph, q):
    return int(list(graph.query(q, initNs=PREFIXES))[0][0])
checks = {
  'things that are an oag:Asset (inheritance: Well/Facility/Field ⊑ Asset)':
      'SELECT (COUNT(DISTINCT ?x) AS ?n) WHERE { ?x a oag:Asset FILTER(STRSTARTS(STR(?x), "https://example.org/oag/id/")) }',
  'well → field links via oag:partOf (sub-property + transitivity through the facility)':
      'SELECT (COUNT(*) AS ?n) WHERE { ?w a oag:Well ; oag:partOf ?f . ?f a oag:Field FILTER(STRSTARTS(STR(?w), "https://example.org/oag/id/")) }',
  'oag:hasPart links (inverse of partOf)':
      'SELECT (COUNT(*) AS ?n) WHERE { ?a oag:hasPart ?b FILTER(STRSTARTS(STR(?a), "https://example.org/oag/id/") && STRSTARTS(STR(?b), "https://example.org/oag/id/")) }',
  'oag:operates links (inverse of operatedBy)':
      'SELECT (COUNT(*) AS ?n) WHERE { ?o oag:operates ?a FILTER(STRSTARTS(STR(?a), "https://example.org/oag/id/")) }',
  'canonical asset ≡ OSDU record (owl:hasKey → owl:sameAs)':
      'SELECT (COUNT(*) AS ?n) WHERE { ?a owl:sameAs ?r FILTER(STRSTARTS(STR(?a), "https://example.org/oag/id/") && STRSTARTS(STR(?r), "https://example.org/oag/osdu-record/")) }',
}
print(f'triples: asserted {len(asserted):,} → after reasoning {len(g):,}')
show([{'inference': k, 'asserted': count(asserted, q), 'inferred': count(g, q)} for k, q in checks.items()])

triples: asserted 469 → after reasoning 1,739


,inference,asserted,inferred
0,things that are an oag:Asset (inheritance: Well/Facility/Field ⊑ Asset),0,27
1,well → field links via oag:partOf (sub-property + transitivity through the facility),0,18
2,oag:hasPart links (inverse of partOf),0,42
3,oag:operates links (inverse of operatedBy),0,24
4,canonical asset ≡ OSDU record (owl:hasKey → owl:sameAs),0,24


The knowledge API (`OntologyGraph`) asks those inferred facts directly. A field's assets, wells and
facilities alike, come from inheritance and the part-of closure; the well's OSDU record comes from the key.

In [10]:
volve = onto.resolve_field('the volve')          # SKOS synonym, any case
print('resolve_field("the volve") →', volve)
print('assets in field', volve, '→', [a.removeprefix(str(ID)) for a in onto.assets_in_field(volve)])
print('field of well 1 →', onto.field_of_well(1), '; facility of well 1 →', onto.facility_of_well(1))
print('operator of well 1 →', onto.operator_of(str(well)))
print('OSDU record of well 1 →', onto.osdu_record_ids(str(well)))

resolve_field("the volve") → 1
assets in field 1 → ['facility/1/Battery', 'facility/2/Battery', 'well/1', 'well/2', 'well/3', 'well/4', 'well/5', 'well/6']
field of well 1 → 1 ; facility of well 1 → 1
operator of well 1 → Equinor
OSDU record of well 1 → ['oag:WELL:1']


### The vocabulary: business language → governed metrics

SKOS concepts carry what words *mean*. A product phrase resolves to the governed metrics that measure it
(*gross liquids* is a grouping with no metric of its own, so it is oil + water through `skos:broader`), and a
business term resolves to its definition: metric, comparison, threshold.

In [11]:
for phrase in ('crude', 'total liquids', 'associated gas', 'unobtainium'):
    print(f'resolve_product({phrase!r:16}) →', onto.resolve_product(phrase))
print()
print('resolve_term("underperforming") →', json.dumps(onto.resolve_term('underperforming'), indent=1, default=str))

resolve_product('crude'         ) → ['actual_oil']
resolve_product('total liquids' ) → ['actual_oil', 'actual_water']
resolve_product('associated gas') → ['actual_gas']
resolve_product('unobtainium'   ) → None

resolve_term("underperforming") → {
 "term": "below_expected",
 "metric": "production_efficiency",
 "comparison": "less_than",
 "threshold_ref": "surveillance_flag_threshold"
}


## 5 · Rules: *below expected*, two ways  <code>rules.ttl · oag_semantic.rules</code>

A business term holds for a well **over a period**, so it is a dated flag record, never a class (ADR 0041).
The numbers come from the governed metrics; the graph only compares them. Two runtimes of the same term:

- **definition as data**: read the term's SKOS definition from the graph and apply it in code;
- **SHACL rule**: `rules.ttl` (`sh:SPARQLRule`) constructs `oag:Flag` records over per-well period records.

Both must equal gold.

In [12]:
print(open(REPO / 'knowledge' / 'ontology' / 'rules.ttl').read().split('oagr:BelowExpectedRule', 1)[1][:1150])

 a sh:NodeShape ;
    sh:targetClass oag:WellPeriod ;
    sh:rule [
        a sh:SPARQLRule ;
        sh:prefixes <https://example.org/oag/rules> ;
        sh:construct """
            CONSTRUCT {
                ?flag a oag:Flag ; oag:ofTerm term:below_expected ; oag:ofWell ?well ;
                      oag:periodStart ?start ; oag:periodEnd ?end ; oag:productionEfficiency ?eff .
            } WHERE {
                $this oag:ofWell ?well ; oag:periodStart ?start ; oag:periodEnd ?end ;
                      oag:actualOil ?actual ; oag:expectedOil ?expected .
                OPTIONAL { $this oag:productionEfficiency ?eff }
                term:below_expected oag:thresholdRef ?ref .
                ?parameter a oag:Parameter ; oag:parameterName ?ref ; oag:value ?threshold .
                FILTER(?actual < ?threshold * ?expected)
                BIND(IRI(CONCAT(STRBEFORE(STR(?well), "well/"), "flag/below_expected/",
                                STRAFTER(STR(?well), "/id/"), "/", STR

In [13]:
from oag_semantic.rules import below_expected_by_definition, below_expected_by_rule, derive_flags
by_definition = below_expected_by_definition(DATA, onto)
by_rule = below_expected_by_rule(DATA, onto)
print('definition as data :', by_definition)
print('SHACL rule         :', by_rule)
print('gold               :', want, '→', 'AGREE' if by_definition == by_rule == want else 'DIFFER')
flags = derive_flags(DATA, onto)
show([{'flag': str(f).removeprefix(str(ID)), 'efficiency': round(float(flags.value(f, OAG.productionEfficiency)), 3)}
      for f in sorted(flags.subjects(OAG.ofTerm, None))])

definition as data : [1, 2, 5, 6, 7, 8, 9, 10, 11, 12, 13, 17]
SHACL rule         : [1, 2, 5, 6, 7, 8, 9, 10, 11, 12, 13, 17]
gold               : [1, 2, 5, 6, 7, 8, 9, 10, 11, 12, 13, 17] → AGREE


,flag,efficiency
0,flag/below_expected/well/1/2025-03-25/2025-03-31,0.361
1,flag/below_expected/well/10/2025-03-25/2025-03-31,0.654
2,flag/below_expected/well/11/2025-03-25/2025-03-31,0.709
3,flag/below_expected/well/12/2025-03-25/2025-03-31,0.612
4,flag/below_expected/well/13/2025-03-25/2025-03-31,0.643
5,flag/below_expected/well/17/2025-03-25/2025-03-31,0.880
6,flag/below_expected/well/2/2025-03-25/2025-03-31,0.436
7,flag/below_expected/well/5/2025-03-25/2025-03-31,0.809
8,flag/below_expected/well/6/2025-03-25/2025-03-31,0.319
9,flag/below_expected/well/7/2025-03-25/2025-03-31,0.681


## 6 · Constraints: SHACL catches bad data before it's answered  <code>shapes.ttl · oag_semantic.shacl</code>

Shapes state what must be true: every well in exactly one field, a well's field equal to its facility's,
no shared keys, and SPARQL-based checks over the daily records (duplicates, flat-lined meters, unit
errors). The generator can plant faults and record them as gold, so the shapes are graded too. A small
dataset with planted faults:

In [14]:
from oag_generator import generate_dataset
from oag_semantic.ontology import map_volumes
from oag_semantic.shacl import data_quality_findings, validate
small = {'seed': 7, 'start_date': '2024-01-01', 'end_date': '2024-02-15', 'n_fields': 2, 'wells_per_field': 3,
         'operators': ['Equinor', 'AkerBP'], 'surveillance_window_days': 7}
knob = {'duplicate_count': 2, 'correction_count': 2, 'flatline_count': 1, 'flatline_days': 3, 'unit_error_count': 2}
DQ = Path(tempfile.mkdtemp()) / 'dq'
generate_dataset({**small, 'data_quality': knob}, DQ)
dq_onto = load_ontology(DQ)
results = validate(dq_onto, map_volumes(DQ))
found = data_quality_findings(DQ, dq_onto)
gold_ids = sorted(f['finding_id'] for f in json.loads((DQ / 'gold' / 'data_quality.json').read_text())['findings'])
print(f'{len(results)} shape results; findings {[f["finding_id"] for f in found] == gold_ids and "= gold" or "≠ gold"}')
show([{'shape': r.shape, 'severity': r.severity, 'focus': r.focus.removeprefix(str(ID))} for r in results]).drop_duplicates('shape')

8 shape results; findings = gold


,shape,severity,focus
0,DuplicateRecordShape,Info,well-day/1/2024-01-27
3,FlatlineShape,Warning,well/2
4,LateCorrectionShape,Info,well-day/2/2024-02-14
6,UnitErrorShape,Warning,well/1


# Part II · One question, end to end

Part I showed each layer on its own. Here one question goes through all of them: the hero question,
from its entry in the question catalog, through what the generator did to make its answer true, the
records that carry that answer, the governed metric and the rule that find it, the gold that records it,
to an answer submitted and graded. Part III then asks the same question of every platform.

## 7 · *Which wells produced below expected oil rate this week, and by how much?*

**The question, in the catalog.** `spec/questions/catalog.yaml` holds every question the system must
answer: its text, the behaviour expected (answer, or decline), the gold artifact, and how it is
graded. Here grading compares a *set* of flagged wells, keyed by `well_id`, on four values.

In [15]:
hero = next(q for q in load_catalog().questions() if q.id == 'surveillance-below-expected-oil')
print(f'id                {hero.id}\ntier              {hero.tier}\nexpected_behavior {hero.expected_behavior}')
print(f'gold              {hero.gold_artifact}\ntext              {hero.text}')
print(f'graded as         set {hero.grading.set_key!r} by {hero.grading.id_key!r} on {list(hero.grading.value_keys)}')

id                surveillance-below-expected-oil
tier              straight
expected_behavior answered
gold              gold/surveillance.json
text              Which wells produced below expected oil rate this week, and by how much?
graded as         set 'flagged' by 'well_id' on ['expected_oil_bbl', 'actual_oil_bbl', 'shortfall_bbl', 'efficiency']


**How the generator makes it true.** For each well it draws an Arps decline curve: the **expected**
oil rate at full uptime, written as the forecast rows of `PRODUCT_VOLUME_SUMMARY`. It then writes the
**actual** daily oil to `WELL_VOL_DAILY` as

> actual = expected × performance × uptime × breakthrough loss

- *performance* is drawn per well from two populations: healthy wells around 1.0, and an impaired 20%
  around 0.72 (well 1 is pinned impaired);
- *uptime* is cut by the downtime events it plants (about one a month per well, each with a cause);
- *breakthrough loss* applies to 35% of wells: after an onset date, water and gas break through and
  oil falls away from the forecast, which is deliberately left untouched.

So a well falls below expected either because it was down, or because it produced less while it was
on. Over the surveillance window, every well, split into those two parts:

In [16]:
W0, W1 = surv.window_start, surv.window_end
uptime = dict(con.sql(f'''SELECT WELL_ID, AVG(HOURS_ON) / 24 FROM well_vol_daily
                          WHERE VOLUME_DATE BETWEEN '{W0}' AND '{W1}' GROUP BY WELL_ID''').fetchall())
flagged_ids = {w.well_id for w in surv.flagged}
show([{'well_id': w.well_id, 'efficiency': round(w.efficiency, 3), 'uptime': round(uptime[w.well_id], 3),
       'efficiency while on': round(w.efficiency / uptime[w.well_id], 3),
       'flagged (< 0.9)': '●' if w.well_id in flagged_ids else ''}
      for w in sorted(surv.wells, key=lambda w: w.efficiency)])

,well_id,efficiency,uptime,efficiency while on,flagged (< 0.9)
0,6,0.319,1.000,0.319,●
1,1,0.361,1.000,0.361,●
2,2,0.436,0.897,0.486,●
3,8,0.469,0.831,0.565,●
4,12,0.612,1.000,0.612,●
5,13,0.643,1.000,0.643,●
6,10,0.654,1.000,0.654,●
7,9,0.668,0.857,0.779,●
8,7,0.681,1.000,0.681,●
9,11,0.709,0.947,0.748,●


**The records behind one answer row.** The worst shortfall, traced through the data model. Actual oil
comes straight from `WELL_VOL_DAILY`. Expected oil is reached through the well's reporting entity, to its
forecast rows in `PRODUCT_VOLUME_SUMMARY`:

In [17]:
top = surv.flagged[0]
daily = con.sql(f'''
    SELECT v.VOLUME_DATE AS day, v.HOURS_ON AS hours_on, p.VOLUME AS expected_oil, v.OIL_VOLUME AS actual_oil
    FROM well_vol_daily v
    JOIN reporting_entity r ON r.REPORTING_ENTITY_KIND = 'Well' AND r.ASSOCIATED_OBJECT_ID = v.WELL_ID
    JOIN product_volume_summary p ON p.REPORTING_ENTITY_ID = r.REPORTING_ENTITY_ID AND p.START_DATE = v.VOLUME_DATE
         AND p.QUANTITY_METHOD = 'Forecast' AND p.PRODUCT = 'Oil'
    WHERE v.WELL_ID = {top.well_id} AND v.VOLUME_DATE BETWEEN '{W0}' AND '{W1}' ORDER BY day''').df()
print(f'well {top.well_id} ({top.uwi}), {W0} .. {W1}')
display(daily)
exp, act = daily.expected_oil.sum(), daily.actual_oil.sum()
print(f'sum: expected {exp:,.1f} bbl, actual {act:,.1f} bbl, shortfall {exp - act:,.1f} bbl, efficiency {act / exp:.3f}')
print(f'the governed metric (§3): expected {top.expected_oil_bbl:,.1f}, actual {top.actual_oil_bbl:,.1f}, '
      f'efficiency {top.efficiency:.3f}')

well 12 (NO 15/9-F-12), 2025-03-25 .. 2025-03-31


,day,hours_on,expected_oil,actual_oil
0,2025-03-25,24.0,11923.534,7414.77
1,2025-03-26,24.0,11900.600,7215.97
2,2025-03-27,24.0,11877.722,7049.00
3,2025-03-28,24.0,11854.901,7724.29
4,2025-03-29,24.0,11832.136,6836.84
5,2025-03-30,24.0,11809.427,7223.75
6,2025-03-31,24.0,11786.774,7329.51


sum: expected 82,985.1 bbl, actual 50,794.1 bbl, shortfall 32,191.0 bbl, efficiency 0.612
the governed metric (§3): expected 82,985.1, actual 50,794.1, efficiency 0.612


**The rule's record of it.** The SHACL rule (§5) turned the same numbers into a dated `oag:Flag` for
the term *below expected*, for this well, over this window:

In [18]:
from rdflib import RDF
flag = next(f for f in flags.subjects(OAG.ofWell, ID[f'well/{top.well_id}']))
for p, o in sorted(flags.predicate_objects(flag)):
    print(f"{fmt(flag)}  {'a' if p == RDF.type else fmt(p)}  {fmt(o)}")

id:flag/below_expected/well/12/2025-03-25/2025-03-31  a  oag:Flag
id:flag/below_expected/well/12/2025-03-25/2025-03-31  oag:ofTerm  id:term/below_expected
id:flag/below_expected/well/12/2025-03-25/2025-03-31  oag:ofWell  id:well/12
id:flag/below_expected/well/12/2025-03-25/2025-03-31  oag:periodEnd  2025-03-31
id:flag/below_expected/well/12/2025-03-25/2025-03-31  oag:periodStart  2025-03-25
id:flag/below_expected/well/12/2025-03-25/2025-03-31  oag:productionEfficiency  0.6120873948760004


**Gold.** What the generator wrote down for this question: the window and threshold it used, the
flagged set, and a reference answer in words. The row for the same well:

In [19]:
g = gold('surveillance')
print(f"window {g['window']['start']} .. {g['window']['end']}, flag below {g['flag_threshold']}, "
      f"{g['n_flagged']} of {g['n_wells_evaluated']} wells flagged")
print(json.dumps(next(r for r in g['flagged'] if r['well_id'] == top.well_id), indent=1))
print('answer:', g['answer'])

window 2025-03-25 .. 2025-03-31, flag below 0.9, 12 of 18 wells flagged
{
 "uwi": "NO 15/9-F-12",
 "well_id": 12,
 "field_id": 2,
 "expected_oil_bbl": 82985.094,
 "actual_oil_bbl": 50794.130000000005,
 "shortfall_bbl": 32190.963999999993,
 "efficiency": 0.6120873948760004
}
answer: 12 of 18 wells produced below 90% of expected oil during 2025-03-25..2025-03-31; NO 15/9-F-12 missed by the most (32191.0 bbl, 61.2% of expected).


**An answer, graded.** Every runtime answers in the same published contract
(`spec/questions/answer_submission.schema.json`): the question id, the answer in words, and
`key_values` holding the graded set. `score_submissions` checks the set by `well_id` and each value
within a relative 1e-6. A right answer, and one that misses a well and gets a shortfall 1% wrong:

In [20]:
from oag_semantic.grading import score_submissions
rows = [{k: getattr(w, k) for k in ('well_id', 'expected_oil_bbl', 'actual_oil_bbl', 'shortfall_bbl', 'efficiency')}
        for w in surv.flagged]
right = {'question_id': hero.id, 'answer': g['answer'], 'key_values': {'flagged': rows}}
print(json.dumps({**right, 'key_values': {'flagged': rows[:1] + [f'… {len(rows) - 1} more']}}, indent=1, ensure_ascii=False))
wrong_rows = [dict(r) for r in rows[1:]]
wrong_rows[0]['shortfall_bbl'] *= 1.01
wrong = {**right, 'key_values': {'flagged': wrong_rows}}
for name, sub in [('right', right), ('wrong', wrong)]:
    grade_ = next(x for x in score_submissions({hero.id: sub}, DATA).grades if x.question_id == hero.id)
    print(f'{name:5}: {grade_.summary()}')

{
 "question_id": "surveillance-below-expected-oil",
 "answer": "12 of 18 wells produced below 90% of expected oil during 2025-03-25..2025-03-31; NO 15/9-F-12 missed by the most (32191.0 bbl, 61.2% of expected).",
 "key_values": {
  "flagged": [
   {
    "well_id": 12,
    "expected_oil_bbl": 82985.094,
    "actual_oil_bbl": 50794.130000000005,
    "shortfall_bbl": 32190.963999999993,
    "efficiency": 0.6120873948760004
   },
   "… 11 more"
  ]
 }
}
right: PASS surveillance-below-expected-oil (12/12 rows)
wrong: FAIL surveillance-below-expected-oil; missing [12]; 1 value mismatch(es)


**Where it goes next.** The same question, in Part III: on GraphDB (§8), compiled by the Snowflake
Semantic View and the Fabric Direct Lake model (§10), and asked in these words of all three agents
(§11), each graded by this same `score_submissions`.

# Part III · How it's instantiated

The same contents, deployed: a second RDF engine, the ontology's knowledge generated onto Snowflake
and Fabric, each platform's own semantic model, and three agents, all checked against the gold from
Part I. Each section needs its runtime (see the note at the top) and is skipped without it.

## 8 · A second engine: GraphDB, with Ontop over DuckDB  <code>oag_semantic.graphdb</code>

Same files, different engine. The asserted graph is loaded into GraphDB (OWL 2 RL ruleset, so **GraphDB
does the reasoning**), the shapes run in GraphDB's SHACL engine, and the daily volumes are never copied:
an Ontop repository serves them live from DuckDB through `volumes.r2rml.ttl`. `master_graph()` wraps
the repository in the same `OntologyGraph` API, so every question runs unchanged.

One finding: GraphDB's own key rule misses some links on a two-property key (the facility), differently
on each load, so the key rule is applied in code before upload on both engines (ADR 0051).

In [21]:
if not runtimes['GraphDB']:
    print('GraphDB is not running (knowledge/graphdb/graphdb.sh up): skipped')
else:
    repos = graphdb.load(DATA, prefix='oagprimer')  # its own repositories: a dataset you loaded as 'oag' is left alone
    remote = graphdb.master_graph(repos['master'])
    print('GraphDB', graphdb.version().get('productVersion'), '→ repositories', repos)
    asks = [('resolve_field', 'the volve'), ('assets_in_field', volve), ('field_of_well', 1),
            ('operator_of', str(well)), ('osdu_record_ids', str(ID['facility/1/Battery'])),
            ('resolve_product', 'total liquids'), ('assets_operated_by', 'equinor')]
    display(show([{'question': f'{m}({a!r})', 'in-process': str(getattr(onto, m)(a))[:70],
                   'same on GraphDB': getattr(onto, m)(a) == getattr(remote, m)(a)} for m, a in asks]))

GraphDB 10.8.14 → repositories {'master': 'oagprimer-master', 'facts': 'oagprimer-facts'}


,question,in-process,same on GraphDB
0,resolve_field('the volve'),1,True
1,assets_in_field(1),"['https://example.org/oag/id/facility/1/Battery', 'https://example.org",True
2,field_of_well(1),1,True
3,operator_of('https://example.org/oag/id/well/1'),Equinor,True
4,osdu_record_ids('https://example.org/oag/id/facility/1/Battery'),['oag:FACILITY:1:Battery'],True
5,resolve_product('total liquids'),"['actual_oil', 'actual_water']",True
6,assets_operated_by('equinor'),"['https://example.org/oag/id/facility/1/Battery', 'https://example.org",True


**Hybrid query**: the hierarchy from the reasoned master graph, joined in one SPARQL query to the
daily volumes Ontop serves live from DuckDB (`SERVICE <repository:oagprimer-facts>`). It equals the governed
rollup.

In [22]:
if runtimes['GraphDB']:
    from oag_semantic.compile import compute_rollup
    rollup = compute_rollup(DATA)
    by_field = graphdb.oil_by_field(rollup.curr_start, rollup.curr_end, prefix='oagprimer')
    display(show([{'field_id': r['field_id'], 'oil this month (GraphDB + Ontop)': round(by_field[r['field_id']], 1),
                   'governed rollup': round(r['oil_curr'], 1)} for r in rollup.by_field]))

,field_id,oil this month (GraphDB + Ontop),governed rollup
0,3,1538397.2,1538397.2
1,2,1057820.5,1057820.5
2,1,326808.7,326808.7


## 9 · The knowledge, generated onto the platforms  <code>oag_semantic.platform_knowledge</code>

The platforms have no OWL reasoner, so the ontology's knowledge is **generated** for them: SQL views for the
class hierarchy (`ENTITY`, `ASSET`), the part-of closure (`PART_OF`), identity on the
keys (`SAME_AS`), and tables for synonyms, terms and products. One generator, two dialects. Below, each
platform's deployed views are **checked live against the ontology** with the same `knowledge_checks`.

First, connections. Snowflake through the `snow` CLI's key-pair connection; Fabric through `az login`
(Entra) for the Warehouse SQL endpoint and the Power BI REST API.

In [23]:
from oag_semantic.platform_knowledge import DIALECTS, knowledge_checks, render

sf = fab = None
if runtimes['Snowflake (snow connection)']:
    import tomllib, snowflake.connector
    _cfg = Path('~/Library/Application Support/snowflake/config.toml').expanduser()
    _p = dict(tomllib.load(open(_cfg, 'rb'))['connections'][os.environ.get('OAG_SNOWFLAKE_CONNECTION', 'proton')])
    _p.pop('database', None); _p.pop('schema', None)
    sf = snowflake.connector.connect(**_p)
if runtimes['Fabric (FABRIC_* env)']:
    import mssql_python
    fab = mssql_python.connect(f"Server={os.environ['FABRIC_SQL_ENDPOINT']};Database={os.environ.get('FABRIC_WAREHOUSE', 'oag_reference')};"
                               'Authentication=ActiveDirectoryDefault;Encrypt=yes', autocommit=True)

def rows(con, sql):
    cur = con.cursor()
    cur.execute(sql)
    return [tuple(r) for r in cur.fetchall()], [d[0] for d in cur.description]

def sql_df(con, sql):
    r, cols = rows(con, sql)
    return pd.DataFrame(r, columns=cols)

print('Snowflake:', sf and sql_df(sf, 'SELECT CURRENT_ROLE() AS role').iloc[0].to_dict())
print('Fabric   :', fab and sql_df(fab, 'SELECT DB_NAME() AS db, SUSER_SNAME() AS who').iloc[0].to_dict()['db'])

Snowflake: {'ROLE': 'SYSADMIN'}
Fabric   : oag_reference


What the generator writes: per platform, the vocabulary tables, the ontology knowledge and the identity
view. An excerpt, the part-of closure (Snowflake dialect): the transitive property becomes joins over
the mapped edges, as deep as the longest chain the mapping allows, so no recursion is needed.

In [24]:
files = render(onto)
show([{'file': str(p.relative_to(REPO)), 'lines': len(t.splitlines())} for p, t in files.items()])

,file,lines
0,platforms/databricks/sql/04_knowledge_layer.sql,149
1,platforms/databricks/sql/06_ontology_knowledge.sql,64
2,platforms/databricks/sql/07_ontology_identity.sql,12
3,platforms/snowflake/sql/04_knowledge_layer.sql,149
4,platforms/snowflake/sql/11_ontology_knowledge.sql,64
5,platforms/snowflake/sql/12_ontology_identity.sql,12
6,platforms/fabric/sql/04_knowledge_tables.sql,60
7,platforms/fabric/sql/06_ontology_knowledge.sql,84
8,platforms/fabric/sql/07_ontology_identity.sql,13


In [25]:
snow_sql = next(t for p, t in files.items() if p.name == '11_ontology_knowledge.sql' and 'snowflake' in str(p))
start = snow_sql.index('-- oag:partOf is transitive')
print(snow_sql[start:snow_sql.index(';', start) + 1])

-- oag:partOf is transitive: the closure, as joins up to the longest chain the mapping allows.
CREATE OR REPLACE VIEW OAG_FABLE.KNOWLEDGE.part_of AS
SELECT DISTINCT part, whole FROM (
SELECT e1.part, e1.whole FROM OAG_FABLE.KNOWLEDGE.part_of_edge e1
UNION ALL
SELECT e1.part, e2.whole FROM OAG_FABLE.KNOWLEDGE.part_of_edge e1 JOIN OAG_FABLE.KNOWLEDGE.part_of_edge e2 ON e1.whole = e2.part
) chains;


**Snowflake**: the generated views in `OAG_FABLE.KNOWLEDGE`, then every check of the views against
the in-process ontology (classes, inheritance, closure, operators, identity, synonyms, terms, products).
An empty problem list means the platform's views give the ontology's answers.

In [26]:
if sf is None:
    print('Snowflake not configured: skipped')
else:
    display(sql_df(sf, """SELECT table_name, table_type FROM OAG_FABLE.INFORMATION_SCHEMA.TABLES
        WHERE table_schema = 'KNOWLEDGE' AND table_name IN ('ONTOLOGY_CLASS','ENTITY','ASSET','PART_OF_EDGE','PART_OF','SAME_AS',
        'PRODUCT','PRODUCT_SYNONYM','PRODUCT_METRIC') ORDER BY 1"""))
    sf.cursor().execute('USE DATABASE OAG_FABLE')
    problems = knowledge_checks(lambda q: rows(sf, q)[0], DIALECTS['snowflake'], onto)
    print('Snowflake knowledge checks: problems =', problems)
    display(sql_df(sf, 'SELECT * FROM OAG_FABLE.KNOWLEDGE.PART_OF WHERE part LIKE \'%well/1\' ORDER BY whole'))

,TABLE_NAME,TABLE_TYPE
0,ASSET,VIEW
1,ENTITY,VIEW
2,ONTOLOGY_CLASS,BASE TABLE
3,PART_OF,VIEW
4,PART_OF_EDGE,VIEW
5,PRODUCT,BASE TABLE
6,PRODUCT_METRIC,BASE TABLE
7,PRODUCT_SYNONYM,BASE TABLE
8,SAME_AS,VIEW


Snowflake knowledge checks: problems = []


,PART,WHOLE
0,https://example.org/oag/id/well/1,https://example.org/oag/id/facility/1/Battery
1,https://example.org/oag/id/well/1,https://example.org/oag/id/field/1


**Fabric**: the same generator's T-SQL views in the Warehouse's `knowledge` schema, the same checks.
Fabric's agent may only read the Direct Lake semantic model, which reads tables, not views, so the answers
are also copied into four `semantic.*` tables for its tools (ADR 0050).

In [27]:
if fab is None:
    print('Fabric not configured: skipped')
else:
    problems = knowledge_checks(lambda q: rows(fab, q)[0], DIALECTS['fabric'], onto)
    print('Fabric knowledge checks: problems =', problems)
    display(sql_df(fab, """SELECT 'asset_hierarchy' AS copied_table, COUNT(*) AS n FROM semantic.asset_hierarchy
        UNION ALL SELECT 'osdu_crosswalk', COUNT(*) FROM semantic.osdu_crosswalk
        UNION ALL SELECT 'product_synonym', COUNT(*) FROM semantic.product_synonym
        UNION ALL SELECT 'product_metric', COUNT(*) FROM semantic.product_metric"""))

Fabric knowledge checks: problems = []


,copied_table,n
0,asset_hierarchy,24
1,osdu_crosswalk,24
2,product_synonym,12
3,product_metric,5


## 10 · The platforms' semantic models reproduce gold

Each platform translates the governed metrics into its own semantic model. The hero question through each:

- **Snowflake**: a Semantic View, queried with `SEMANTIC_VIEW()` (dimensions + metrics, no SQL joins);
- **Fabric**: the Direct Lake semantic model (TMDL measures), queried in DAX through `executeQueries`;
- **Databricks**: metric views are written (`platforms/databricks/sql`) but not deployed in this workspace.

In [28]:
if sf is not None:
    sv = sql_df(sf, f"""SELECT well_id, actual_oil, expected_oil, production_efficiency
        FROM SEMANTIC_VIEW(OAG_FABLE.SEMANTIC.SURVEILLANCE
          DIMENSIONS surv.well_id METRICS surv.expected_oil, surv.actual_oil, surv.production_efficiency
          WHERE surv.volume_date BETWEEN '{surv.window_start}' AND '{surv.window_end}')
        WHERE production_efficiency < 0.9 ORDER BY well_id""")
    print('Snowflake Semantic View flagged:', sorted(sv.WELL_ID.astype(int)), '→', '= gold' if sorted(sv.WELL_ID.astype(int)) == want else '≠ gold')
    display(sv.head(4))

Snowflake Semantic View flagged: [1, 2, 5, 6, 7, 8, 9, 10, 11, 12, 13, 17] → = gold


,WELL_ID,ACTUAL_OIL,EXPECTED_OIL,PRODUCTION_EFFICIENCY
0,1,4686.85,12994.179,0.360688
1,2,17117.53,39240.377,0.436222
2,5,12806.62,15822.590,0.809388
3,6,3961.53,12421.417,0.318927


In [29]:
if fab is not None:
    import fabric
    model_id = fabric.item_id('SemanticModel', os.environ.get('FABRIC_SEMANTIC_MODEL', 'oag_reference_model'))
    print(fabric.THEME_QUERIES['surveillance'].strip()[:400])
    dax_rows = fabric.dax(model_id, fabric.theme_query('surveillance'))
    got = sorted(int(r['well_id']) for r in dax_rows)
    print('\nFabric DAX flagged:', got, '→', '= gold' if got == want else '≠ gold')

EVALUATE
FILTER(
    ADDCOLUMNS(CALCULATETABLE(VALUES(well[well_id]){field_filter}), "uwi", CALCULATE(SELECTEDVALUE(well[uwi])),
        "expected_oil_bbl", [Surveillance Expected Oil], "actual_oil_bbl", [Surveillance Actual Oil],
        "shortfall_bbl", [Surveillance Shortfall], "efficiency", [Surveillance Efficiency], "flag", [Is Below Expected]),
    [flag] = 1)



Fabric DAX flagged: [1, 2, 5, 6, 7, 8, 9, 10, 11, 12, 13, 17] → = gold


Fabric also has its own **Fabric IQ ontology** item (entities, relationships, rules), deployed from
`platforms/fabric/ontology` and queried over MCP in natural language. Its custom agent uses it to resolve
names, as here:

In [30]:
fabric_agent = None
if fab is not None and runtimes['Foundry (OAG_FOUNDRY_ENDPOINT)']:
    from agent import ProductionAgent
    fabric_agent = ProductionAgent()
    try:
        print('Fabric IQ ontology: resolve_field("the volve") →', fabric_agent.resolve_field('the volve'))
    except TimeoutError:
        print('Fabric IQ ontology: the MCP call timed out this run')

Fabric IQ ontology: resolve_field("the volve") → {'field_id': 1, 'field_name': 'Volve'}


## 11 · The agents, asked live and graded against gold

Three agents, the same nine catalog questions (six use cases + three asset-knowledge questions), graded by
the same `score_submissions`. Each agent answers through **governed tools**: numbers only ever come from the
semantic layer, and the graded rows are built in code from the tool's result, never from the model's prose.

| agent | model | knowledge it reasons with |
|---|---|---|
| ontology agent (`oag_semantic.ontology_agent`) | Claude Sonnet 5.5 | the OWL graph, live (in-process; `--graphdb` for GraphDB), plus read-only SPARQL |
| Snowflake `PRODUCTION_AGENT` | Cortex Agent, over REST `:run` | generated `KNOWLEDGE` views, through owner's-rights tools |
| Fabric custom agent | gpt-6-luna on Foundry | the Direct Lake model's copied tables, plus the Fabric IQ ontology |

In [31]:
from oag_generator.questions import load_catalog
from oag_semantic.grading import score_submissions
QUESTIONS = [q for q in load_catalog().questions()
             if q.scenario is None and (GOLD.parent / q.gold_artifact).exists()]

def grade(submissions):
    report = score_submissions(submissions, DATA, knowledge=True)
    return {g.question_id: g for g in report.grades}

scores = {}
print(len(QUESTIONS), "questions, asked in the catalog's own words:")
for q in QUESTIONS:
    print(f' - {q.id}: {q.text}')

9 questions, asked in the catalog's own words:
 - surveillance-below-expected-oil: Which wells produced below expected oil rate this week, and by how much?
 - deferment-top-downtime-causes: What did we defer last month, and what were the top downtime causes?
 - decline-12-month-field-oil: What is the 12-month oil decline for Field X, and which wells are declining faster than forecast?
 - welltest-stale-or-anomalous-allocation: Which wells have stale tests or anomalous allocation?
 - watchlist-down-watering-out-gor-change: Which wells are down, watering out, or showing a GOR change?
 - rollup-oil-gas-water-by-field-operator: Oil/gas/water by field and operator this month vs last -- who are the biggest movers?
 - knowledge-assets-by-field: List every asset in each field, wells and facilities alike, with the field it belongs to.
 - knowledge-gross-liquids-by-field: How much gross liquids did each field produce this month?
 - knowledge-osdu-record-identity: Which canonical well or facility

**The ontology agent** (Claude). For each question: which tools it chose, and whether the graded
answer equals gold.

In [32]:
if not runtimes['Claude (ANTHROPIC_API_KEY)']:
    print('ANTHROPIC_API_KEY not set: skipped')
else:
    from oag_semantic.ontology_agent import OntologyAgent, reasoning_checks
    oa = OntologyAgent(DATA)
    subs, tools = {}, {}
    for q in QUESTIONS:
        run = oa.ask(q.text)
        tools[q.id] = [n for n, _ in run.tool_calls]
        if run.submission:
            subs[run.submission['question_id']] = run.submission
    g = grade(subs)
    scores['ontology agent'] = g
    display(show([{'question': q.id, 'tools': ', '.join(tools[q.id]), 'correct': g[q.id].correct, 'note': '' if g[q.id].correct else g[q.id].summary()[:160]} for q in QUESTIONS]))

,question,tools,correct,note
0,surveillance-below-expected-oil,surveillance_below_expected,True,
1,deferment-top-downtime-causes,deferment_by_cause,True,
2,decline-12-month-field-oil,decline_wells_faster,True,
3,welltest-stale-or-anomalous-allocation,welltest_flags,True,
4,watchlist-down-watering-out-gor-change,watchlist_flags,True,
5,rollup-oil-gas-water-by-field-operator,"rollup_by_field_and_operator, sparql_query",True,
6,knowledge-assets-by-field,asset_hierarchy,True,
7,knowledge-gross-liquids-by-field,product_volume_by_field,True,
8,knowledge-osdu-record-identity,osdu_crosswalk,True,


And the questions only reasoning answers: entity questions graded against answers computed from the
canonical tables directly. *List every asset in a field* needs inheritance and the part-of closure; *which
well is OSDU record …* needs the key rule.

In [33]:
if runtimes['Claude (ANTHROPIC_API_KEY)']:
    out = []
    for question, expected in reasoning_checks(DATA):
        run = oa.ask(question)
        out.append({'question': question, 'tools': ', '.join(n for n, _ in run.tool_calls),
                    'found': f'{len(run.entities or [])}/{len(expected)}', 'correct': run.entities == expected})
    display(show(out))
    print('answer to the last one:', run.text[:300])

,question,tools,found,correct
0,"List every asset in the Volve field, wells and facilities alike.","resolve_field, assets_in_field, report_entities",8/8,True
1,Which wells in the Volve field are operated by Equinor?,"resolve_field, assets_operated_by, assets_in_field, report_entities",6/6,True
2,Which facilities does Equinor operate?,"assets_operated_by, report_entities",2/2,True
3,Which products make up total liquids?,"resolve_product, report_entities",2/2,True
4,Which well is OSDU record oag:WELL:18?,"sparql_query, sparql_query, osdu_record_ids, osdu_crosswalk, sparql_query, report_enti...",1/1,True


answer to the last one: OSDU record `oag:WELL:18` is the well **Gullfaks-6**. Its canonical IRI is `https://example.org/oag/id/well/18`.

- **UWI:** NO 15/9-F-18
- **Facility:** Battery 6 (facility/6)
- **Field:** field/3
- **Operator:** Wintershall


**The Snowflake Cortex Agent**, deployed as `OAG_FABLE.KNOWLEDGE.PRODUCTION_AGENT` and called over its
REST `:run` endpoint with a key-pair JWT. `to_submission` adapts its tool results to the answer contract.

In [34]:
if sf is not None:
    import importlib.util
    sys.path.append(str(REPO / 'platforms' / 'snowflake' / 'agent'))  # its sfconfig helper; appended, so Fabric's agent.py still wins
    _spec = importlib.util.spec_from_file_location('sf_rest', REPO / 'platforms' / 'snowflake' / 'agent' / 'test_agent_rest.py')
    sf_rest = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(sf_rest)
    jwt = sf_rest._jwt()
    by_qid = {t['question_id']: t for t in [*sf_rest.THEMES.values(), *sf_rest.KNOWLEDGE.values()]}
    subs, tools = {}, {}
    for q in QUESTIONS:
        t = by_qid[q.id]
        resp = sf_rest.run_agent(q.text, jwt)
        tools[q.id] = [i['tool_result'].get('name') for i in resp.get('content', []) if i['type'] == 'tool_result']
        sub = sf_rest.to_submission(resp, t)
        if sub['question_id'] != 'unresolved':
            subs[sub['question_id']] = sub
    g = grade(subs)
    scores['Snowflake Cortex Agent'] = g
    display(show([{'question': q.id, 'tools': ', '.join(filter(None, tools[q.id])), 'correct': g[q.id].correct, 'note': '' if g[q.id].correct else g[q.id].summary()[:160]} for q in QUESTIONS]))

,question,tools,correct,note
0,surveillance-below-expected-oil,surveillance_below_expected,True,
1,deferment-top-downtime-causes,"deferment_by_cause, deferment_summary",True,
2,decline-12-month-field-oil,decline_wells_faster,True,
3,welltest-stale-or-anomalous-allocation,welltest_flags,True,
4,watchlist-down-watering-out-gor-change,watchlist_flags,True,
5,rollup-oil-gas-water-by-field-operator,"rollup_by_operator, rollup_by_field",True,
6,knowledge-assets-by-field,asset_hierarchy,True,
7,knowledge-gross-liquids-by-field,product_volume_by_field,True,
8,knowledge-osdu-record-identity,osdu_crosswalk,True,


**The Fabric custom agent**: gpt-6-luna on Azure AI Foundry, with tools that run fixed DAX queries
against the Direct Lake model (no user text ever reaches a query) and the Fabric IQ ontology for names.

In [35]:
if fabric_agent is not None:
    subs = {}
    import urllib.error
    for q in QUESTIONS:
        try:
            sub = fabric_agent.answer(q.text)
        except urllib.error.URLError:  # a dropped connection: ask once more
            sub = fabric_agent.answer(q.text)
        subs[sub['question_id']] = sub
    g = grade(subs)
    scores['Fabric custom agent'] = g
    display(show([{'question': q.id, 'behavior': subs.get(q.id, {}).get('behavior'), 'correct': g[q.id].correct, 'note': '' if g[q.id].correct else g[q.id].summary()[:160]}
                  for q in QUESTIONS]))

,question,behavior,correct,note
0,surveillance-below-expected-oil,answered,True,
1,deferment-top-downtime-causes,answered,True,
2,decline-12-month-field-oil,answered,True,
3,welltest-stale-or-anomalous-allocation,answered,True,
4,watchlist-down-watering-out-gor-change,answered,True,
5,rollup-oil-gas-water-by-field-operator,answered,True,
6,knowledge-assets-by-field,answered,True,
7,knowledge-gross-liquids-by-field,answered,True,
8,knowledge-osdu-record-identity,answered,True,


## Scorecard

Correct answers per agent, use cases and asset knowledge separately (theme 8 is graded apart, ADR 0048).

In [36]:
knowledge_ids = {q.id for q in QUESTIONS if q.id.startswith('knowledge-')}
show([{'agent': a, 'use cases (6)': sum(g[q].correct for q in g if q not in knowledge_ids and q in {x.id for x in QUESTIONS}),
       'asset knowledge (3)': sum(g[q].correct for q in knowledge_ids)} for a, g in scores.items()])

,agent,use cases (6),asset knowledge (3)
0,ontology agent,6,3
1,Snowflake Cortex Agent,6,3
2,Fabric custom agent,6,3


## What it demonstrates

- **One definition of each metric** reproduces gold on every platform that was deployed: the governed
  layer, not the model, decides the numbers.
- **The ontology adds what no platform carries natively**: inheritance, a transitive roll-up, inverses,
  identity across systems from keys, and business language mapped to governed metrics, all *inferred*
  from four standard files.
- **Rules and constraints are data**: *below expected* agrees as a definition and as a SHACL rule; SHACL
  shapes find exactly the planted faults.
- **Standards are portable**: GraphDB gives the in-process answers, and the agent scores the same on it.
  Where an engine's rule can't be relied on (owlrl's and GraphDB's key rules), it's applied in code.
- **Generated knowledge travels**: the platforms get the ontology's answers as views, checked against the
  ontology, and their agents answer the asset-knowledge questions from them.

See the [capability matrix](../docs/capability-matrix.md) for the cell-by-cell comparison, and
`docs/adr/` for every decision.

In [37]:
for c in (sf, fab):
    if c is not None:
        c.close()